# 03 — Data Cleaning and Standardisation

## Objective

This notebook cleans and standardises the Olist datasets while preserving the original raw data.

The main tasks include:

- converting date columns;
- handling missing and untranslated product categories;
- preparing one-row-per-ZIP-code geolocation data;
- checking invalid values and timestamp sequences;
- saving validated datasets to the processed data directory.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()

if working_dir.name == "notebooks":
    project_root = working_dir.parent
else:
    project_root = working_dir

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

processed_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

if not raw_dir.exists():
    raise FileNotFoundError(f"Raw data directory not found: {raw_dir}")

print("Project root:", project_root)
print("Raw data directory:", raw_dir)
print("Processed data directory:", processed_dir)

Project root: /Users/liyang/Documents/olist-growth-operations-analytics
Raw data directory: /Users/liyang/Documents/olist-growth-operations-analytics/data/raw
Processed data directory: /Users/liyang/Documents/olist-growth-operations-analytics/data/processed


In [3]:
file_map = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    file_name
    for file_name in file_map.values()
    if not (raw_dir / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing raw data files: {missing_files}"
    )

print(f"All {len(file_map)} raw files were found.")

All 9 raw files were found.


In [4]:
raw_tables = {}

for table_name, file_name in file_map.items():
    raw_tables[table_name] = pd.read_csv(raw_dir / file_name)

    rows, columns = raw_tables[table_name].shape
    print(f"{table_name:<22}" f"rows={rows:>9,}" f"columns={columns}")

customers             rows=   99,441columns=5
geolocation           rows=1,000,163columns=5
order_items           rows=  112,650columns=7
payments              rows=  103,886columns=5
reviews               rows=   99,224columns=7
orders                rows=   99,441columns=8
products              rows=   32,951columns=9
sellers               rows=    3,095columns=4
category_translation  rows=       71columns=2


In [5]:
clean_tables = {
    table_name: df.copy()
    for table_name, df in raw_tables.items()
}

print("Raw tables:", len(raw_tables))
print("Cleaning copies:", len(clean_tables))


Raw tables: 9
Cleaning copies: 9


In [6]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

total_date_columns = sum(
    len(columns)
    for columns in date_columns.values()
)

print("Number of date columns:", total_date_columns)

Number of date columns: 8


In [7]:
date_conversion_records = []

for table_name, columns in date_columns.items():
    for column_name in columns:
        original_series = clean_tables[table_name][column_name]

        missing_before = int(
            original_series.isna().sum()
        )

        converted_series = pd.to_datetime(
            original_series,
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce",
        )

        parse_failure_mask = (
            original_series.notna()
            & converted_series.isna()
        )

        parse_failures = int(
            parse_failure_mask.sum()
        )

        clean_tables[table_name][column_name] = (
            converted_series
        )

        date_conversion_records.append({
            "table_name": table_name,
            "column_name": column_name,
            "missing_before": missing_before,
            "missing_after": int(
                converted_series.isna().sum()
            ),
            "parse_failures": parse_failures,
            "minimum_date": converted_series.min(),
            "maximum_date": converted_series.max(),
        })

date_conversion_df = pd.DataFrame(
    date_conversion_records
)

display(date_conversion_df)

,table_name,column_name,missing_before,missing_after,parse_failures,minimum_date,maximum_date
0,orders,order_purchase_timestamp,0,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,orders,order_approved_at,160,160,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,orders,order_delivered_carrier_date,1783,1783,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,orders,order_delivered_customer_date,2965,2965,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,orders,order_estimated_delivery_date,0,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00
5,order_items,shipping_limit_date,0,0,0,2016-09-19 00:15:34,2020-04-09 22:35:08
6,reviews,review_creation_date,0,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00
7,reviews,review_answer_timestamp,0,0,0,2016-10-07 18:32:28,2018-10-29 12:27:35


In [8]:
for table_name, columns in date_columns.items():
    print(f"\n{table_name}")

    for column_name in columns:
        data_type = clean_tables[table_name][
            column_name
        ].dtype

        print(f"  {column_name}: {data_type}")


orders
  order_purchase_timestamp: datetime64[us]
  order_approved_at: datetime64[us]
  order_delivered_carrier_date: datetime64[us]
  order_delivered_customer_date: datetime64[us]
  order_estimated_delivery_date: datetime64[us]

order_items
  shipping_limit_date: datetime64[us]

reviews
  review_creation_date: datetime64[us]
  review_answer_timestamp: datetime64[us]


In [9]:
date_conversion_path = (
    reports_dir / "date_conversion_checks.csv"
)

date_conversion_df.to_csv(
    date_conversion_path,
    index=False,
    encoding="utf-8",
)

print("Date conversion report saved to:")
print(date_conversion_path)

Date conversion report saved to:
/Users/liyang/Documents/olist-growth-operations-analytics/reports/date_conversion_checks.csv


In [10]:
zip_code_columns = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}

zip_check_records = []

for table_name, column_name in zip_code_columns.items():
    original_series = clean_tables[table_name][
        column_name
    ]

    numeric_series = pd.to_numeric(
        original_series,
        errors="coerce",
    )

    invalid_parse_values = int(
        (
            original_series.notna()
            & numeric_series.isna()
        ).sum()
    )

    cleaned_zip = (
        numeric_series
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    invalid_length_values = int(
        (
            cleaned_zip.dropna().str.len() != 5
        ).sum()
    )

    clean_tables[table_name][column_name] = (
        cleaned_zip
    )

    zip_check_records.append({
        "table_name": table_name,
        "column_name": column_name,
        "missing_values": int(
            cleaned_zip.isna().sum()
        ),
        "invalid_parse_values": invalid_parse_values,
        "invalid_length_values": invalid_length_values,
        "data_type": str(cleaned_zip.dtype),
    })

zip_checks_df = pd.DataFrame(zip_check_records)

display(zip_checks_df)

,table_name,column_name,missing_values,invalid_parse_values,invalid_length_values,data_type
0,customers,customer_zip_code_prefix,0,0,0,string
1,sellers,seller_zip_code_prefix,0,0,0,string
2,geolocation,geolocation_zip_code_prefix,0,0,0,string


In [11]:
text_conversion_records = []

for table_name, df in clean_tables.items():
    for column_name in df.columns:
        series = df[column_name]

        is_text_column = (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
        )

        if not is_text_column:
            continue

        missing_before = int(series.isna().sum())

        cleaned_text = (
            series
            .astype("string")
            .str.strip()
        )

        blank_strings = int(
            cleaned_text.eq("")
            .fillna(False)
            .sum()
        )

        cleaned_text = cleaned_text.replace(
            "",
            pd.NA,
        )

        clean_tables[table_name][column_name] = (
            cleaned_text
        )

        text_conversion_records.append({
            "table_name": table_name,
            "column_name": column_name,
            "missing_before": missing_before,
            "blank_strings_removed": blank_strings,
            "missing_after": int(
                cleaned_text.isna().sum()
            ),
            "data_type_after": str(
                cleaned_text.dtype
            ),
        })

text_conversion_df = pd.DataFrame(
    text_conversion_records
)

print(
    "Text columns standardised:",
    len(text_conversion_df),
)

display(text_conversion_df)

Text columns standardised: 28


,table_name,column_name,missing_before,blank_strings_removed,missing_after,data_type_after
0,customers,customer_id,0,0,0,string
1,customers,customer_unique_id,0,0,0,string
2,customers,customer_zip_code_prefix,0,0,0,string
3,customers,customer_city,0,0,0,string
4,customers,customer_state,0,0,0,string
5,geolocation,geolocation_zip_code_prefix,0,0,0,string
6,geolocation,geolocation_city,0,0,0,string
7,geolocation,geolocation_state,0,0,0,string
8,order_items,order_id,0,0,0,string
9,order_items,product_id,0,0,0,string


In [12]:
data_type_records = []

for table_name, df in clean_tables.items():
    for column_name in df.columns:
        data_type_records.append({
            "table_name": table_name,
            "column_name": column_name,
            "data_type": str(
                df[column_name].dtype
            ),
            "missing_values": int(
                df[column_name].isna().sum()
            ),
        })

data_type_checks_df = pd.DataFrame(
    data_type_records
)

data_type_checks_path = (
    reports_dir / "data_type_checks.csv"
)

data_type_checks_df.to_csv(
    data_type_checks_path,
    index=False,
    encoding="utf-8",
)

print("Data type report saved to:")
print(data_type_checks_path)

Data type report saved to:
/Users/liyang/Documents/olist-growth-operations-analytics/reports/data_type_checks.csv


In [14]:
products_before = clean_tables["products"].copy()

translation_before = clean_tables[
    "category_translation"
].copy()

translated_categories = set(
    translation_before[
        "product_category_name"
    ].dropna()
)

missing_categories_before = int(
    products_before[
        "product_category_name"
    ].isna().sum()
)

untranslated_mask_before = (
    products_before[
        "product_category_name"
    ].notna()
    & ~products_before[
        "product_category_name"
    ].isin(translated_categories)
)

untranslated_products_before = int(
    untranslated_mask_before.sum()
)

print(
    "Missing categories before cleaning:",
    missing_categories_before,
)

print(
    "Untranslated products before cleaning:",
    untranslated_products_before,
)

Missing categories before cleaning: 610
Untranslated products before cleaning: 13


In [15]:
manual_translations = pd.DataFrame({
    "product_category_name": [
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer",
        "unknown",
    ],
    "product_category_name_english": [
        "portable_kitchen_and_food_preparation_appliances",
        "gaming_pc",
        "unknown",
    ],
})

translation_clean = pd.concat(
    [
        translation_before,
        manual_translations,
    ],
    ignore_index=True,
)

duplicate_translation_keys = int(
    translation_clean[
        "product_category_name"
    ].duplicated().sum()
)

print(
    "Translation rows before:",
    len(translation_before),
)

print(
    "Translation rows after:",
    len(translation_clean),
)

print(
    "Duplicate translation keys:",
    duplicate_translation_keys,
)

Translation rows before: 71
Translation rows after: 74
Duplicate translation keys: 0


In [16]:
products_clean = products_before.copy()

products_clean[
    "product_category_name"
] = products_clean[
    "product_category_name"
].fillna("unknown")

product_rows_before_merge = len(products_clean)

products_clean = products_clean.merge(
    translation_clean,
    on="product_category_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

product_rows_after_merge = len(products_clean)

unmatched_after_merge = int(
    (products_clean["_merge"] != "both").sum()
)

missing_english_after = int(
    products_clean[
        "product_category_name_english"
    ].isna().sum()
)

print(
    "Product rows before merge:",
    product_rows_before_merge,
)

print(
    "Product rows after merge:",
    product_rows_after_merge,
)

print(
    "Unmatched products after merge:",
    unmatched_after_merge,
)

print(
    "Missing English categories after merge:",
    missing_english_after,
)

Product rows before merge: 32951
Product rows after merge: 32951
Unmatched products after merge: 0
Missing English categories after merge: 0


In [17]:
products_clean = products_clean.drop(
    columns="_merge"
)

clean_tables["products"] = products_clean

clean_tables["category_translation"] = translation_clean
print("Clean products shape:", clean_tables["products"].shape,)
print("Clean translations shape:", clean_tables["category_translation"].shape,)

Clean products shape: (32951, 10)
Clean translations shape: (74, 2)


In [18]:
assert product_rows_before_merge == product_rows_after_merge
assert duplicate_translation_keys == 0
assert unmatched_after_merge == 0
assert missing_english_after == 0
print("Product category cleaning checks passed.")


Product category cleaning checks passed.


In [19]:
category_cleaning_summary_df = pd.DataFrame({
    "metric": [
        "missing_categories_before",
        "untranslated_products_before",
        "translation_rows_before",
        "translation_rows_after",
        "product_rows_before_merge",
        "product_rows_after_merge",
        "unmatched_products_after_merge",
        "missing_english_categories_after",
    ],
    "value": [
        missing_categories_before,
        untranslated_products_before,
        len(translation_before),
        len(translation_clean),
        product_rows_before_merge,
        product_rows_after_merge,
        unmatched_after_merge,
        missing_english_after,
    ],
})

category_report_path = (
    reports_dir / "category_cleaning_checks.csv"
)

category_cleaning_summary_df.to_csv(
    category_report_path,
    index=False,
    encoding="utf-8",
)

print("Category cleaning report saved to:")
print(category_report_path)


Category cleaning report saved to:
/Users/liyang/Documents/olist-growth-operations-analytics/reports/category_cleaning_checks.csv


In [20]:
geolocation_before = clean_tables[
    "geolocation"
].copy()

geolocation_rows_before = len(
    geolocation_before
)

exact_duplicate_rows = int(
    geolocation_before.duplicated().sum()
)

geolocation_deduplicated = (
    geolocation_before
    .drop_duplicates()
    .copy()
)

geolocation_rows_after_deduplication = len(
    geolocation_deduplicated
)

print(
    "Rows before cleaning:",
    geolocation_rows_before,
)

print(
    "Exact duplicate rows:",
    exact_duplicate_rows,
)

print(
    "Rows after removing exact duplicates:",
    geolocation_rows_after_deduplication,
)

Rows before cleaning: 1000163
Exact duplicate rows: 261831
Rows after removing exact duplicates: 738332


In [21]:
valid_coordinate_mask = (
    geolocation_deduplicated[
        "geolocation_lat"
    ].between(-90, 90)
    & geolocation_deduplicated[
        "geolocation_lng"
    ].between(-180, 180)
)

invalid_coordinate_rows = int(
    (~valid_coordinate_mask).sum()
)

geolocation_valid = (
    geolocation_deduplicated
    .loc[valid_coordinate_mask]
    .copy()
)

print(
    "Invalid coordinate rows:",
    invalid_coordinate_rows,
)

print(
    "Rows with valid coordinates:",
    len(geolocation_valid),
)

Invalid coordinate rows: 0
Rows with valid coordinates: 738332


In [25]:
def most_frequent_value(series):
    modes = series.dropna().mode()

    if modes.empty:
        return pd.NA
    return sorted(modes.astype(str).tolist())[0]
    
geolocation_clean = (
    geolocation_valid
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False,
    )
    .agg(
        geolocation_lat=(
            "geolocation_lat",
            "median",
        ),
        geolocation_lng=(
            "geolocation_lng",
            "median",
        ),
        geolocation_city=(
            "geolocation_city",
            most_frequent_value,
        ),
        geolocation_state=(
            "geolocation_state",
            most_frequent_value,
        ),
        geolocation_observation_count=(
            "geolocation_lat",
            "size",
        ),
    )
)

geolocation_clean[
    "geolocation_lat"
] = geolocation_clean[
    "geolocation_lat"
].round(6)

geolocation_clean[
    "geolocation_lng"
] = geolocation_clean[
    "geolocation_lng"
].round(6)



In [26]:
unique_zip_codes_before = int(
    geolocation_valid[
        "geolocation_zip_code_prefix"
    ].nunique()
)

duplicate_zip_codes_after = int(
    geolocation_clean[
        "geolocation_zip_code_prefix"
    ].duplicated().sum()
)

print(
    "Unique ZIP-code prefixes before aggregation:",
    unique_zip_codes_before,
)

print(
    "Rows after aggregation:",
    len(geolocation_clean),
)

print(
    "Duplicate ZIP-code prefixes after aggregation:",
    duplicate_zip_codes_after,
)

Unique ZIP-code prefixes before aggregation: 19015
Rows after aggregation: 19015
Duplicate ZIP-code prefixes after aggregation: 0


In [27]:
geolocation_zip_values = set(
    geolocation_clean[
        "geolocation_zip_code_prefix"
    ]
)

zip_relationships = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
}

zip_coverage_records = []

for table_name, column_name in zip_relationships.items():
    zip_series = clean_tables[
        table_name
    ][column_name]

    non_null_zip = zip_series.dropna()

    unmatched_mask = ~non_null_zip.isin(
        geolocation_zip_values
    )

    unmatched_rows = int(
        unmatched_mask.sum()
    )

    coverage_pct = (
        (
            len(non_null_zip) - unmatched_rows
        )
        / len(non_null_zip)
        * 100
    )

    zip_coverage_records.append({
        "table_name": table_name,
        "zip_column": column_name,
        "total_rows": len(zip_series),
        "unmatched_rows": unmatched_rows,
        "unmatched_unique_zip_codes": int(
            non_null_zip[
                unmatched_mask
            ].nunique()
        ),
        "coverage_pct": round(
            coverage_pct,
            2,
        ),
    })

zip_coverage_df = pd.DataFrame(
    zip_coverage_records
)

display(zip_coverage_df)

,table_name,zip_column,total_rows,unmatched_rows,unmatched_unique_zip_codes,coverage_pct
0,customers,customer_zip_code_prefix,99441,278,157,99.72
1,sellers,seller_zip_code_prefix,3095,7,7,99.77


In [28]:
orders_clean = clean_tables["orders"]

order_status_df = (
    orders_clean["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

order_status_df["order_pct"] = (
    order_status_df["order_count"]
    / len(orders_clean)
    * 100
).round(2)

display(order_status_df)

,order_status,order_count,order_pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.3
6,created,5,0.01
7,approved,2,0.0


In [29]:
order_timestamp_checks = {
    "approval_before_purchase": (
        orders_clean["order_approved_at"].notna()
        & (
            orders_clean["order_approved_at"]
            < orders_clean["order_purchase_timestamp"]
        )
    ),

    "carrier_before_purchase": (
        orders_clean[
            "order_delivered_carrier_date"
        ].notna()
        & (
            orders_clean[
                "order_delivered_carrier_date"
            ]
            < orders_clean[
                "order_purchase_timestamp"
            ]
        )
    ),

    "carrier_before_approval": (
        orders_clean[
            "order_delivered_carrier_date"
        ].notna()
        & orders_clean["order_approved_at"].notna()
        & (
            orders_clean[
                "order_delivered_carrier_date"
            ]
            < orders_clean["order_approved_at"]
        )
    ),

    "delivery_before_carrier": (
        orders_clean[
            "order_delivered_customer_date"
        ].notna()
        & orders_clean[
            "order_delivered_carrier_date"
        ].notna()
        & (
            orders_clean[
                "order_delivered_customer_date"
            ]
            < orders_clean[
                "order_delivered_carrier_date"
            ]
        )
    ),

    "delivery_before_purchase": (
        orders_clean[
            "order_delivered_customer_date"
        ].notna()
        & (
            orders_clean[
                "order_delivered_customer_date"
            ]
            < orders_clean[
                "order_purchase_timestamp"
            ]
        )
    ),

    "estimated_before_purchase": (
        orders_clean[
            "order_estimated_delivery_date"
        ].notna()
        & (
            orders_clean[
                "order_estimated_delivery_date"
            ]
            < orders_clean[
                "order_purchase_timestamp"
            ]
        )
    ),

    "delivered_status_missing_delivery_date": (
        orders_clean["order_status"].eq("delivered")
        & orders_clean[
            "order_delivered_customer_date"
        ].isna()
    ),

    "non_delivered_status_with_delivery_date": (
        ~orders_clean["order_status"].eq("delivered")
        & orders_clean[
            "order_delivered_customer_date"
        ].notna()
    ),
}

In [30]:
order_timestamp_checks_df = pd.DataFrame({
    "check_name": list(
        order_timestamp_checks.keys()
    ),
    "affected_rows": [
        int(mask.fillna(False).sum())
        for mask in order_timestamp_checks.values()
    ],
})

display(order_timestamp_checks_df)

,check_name,affected_rows
0,approval_before_purchase,0
1,carrier_before_purchase,166
2,carrier_before_approval,1359
3,delivery_before_carrier,23
4,delivery_before_purchase,0
5,estimated_before_purchase,0
6,delivered_status_missing_delivery_date,8
7,non_delivered_status_with_delivery_date,6


In [31]:
numeric_columns = [
    ("order_items", "price"),
    ("order_items", "freight_value"),
    ("payments", "payment_value"),
    ("payments", "payment_installments"),
    ("reviews", "review_score"),
    ("products", "product_weight_g"),
    ("products", "product_length_cm"),
    ("products", "product_height_cm"),
    ("products", "product_width_cm"),
]

numeric_check_records = []

for table_name, column_name in numeric_columns:
    series = pd.to_numeric(
        clean_tables[table_name][column_name],
        errors="coerce",
    )

    numeric_check_records.append({
        "table_name": table_name,
        "column_name": column_name,
        "missing_values": int(series.isna().sum()),
        "negative_values": int((series < 0).sum()),
        "zero_values": int((series == 0).sum()),
        "minimum_value": series.min(),
        "maximum_value": series.max(),
    })

numeric_checks_df = pd.DataFrame(
    numeric_check_records
)

display(numeric_checks_df)

,table_name,column_name,missing_values,negative_values,zero_values,minimum_value,maximum_value
0,order_items,price,0,0,0,0.85,6735.00
1,order_items,freight_value,0,0,383,0.00,409.68
2,payments,payment_value,0,0,9,0.00,13664.08
3,payments,payment_installments,0,0,2,0.00,24.00
4,reviews,review_score,0,0,0,1.00,5.00
5,products,product_weight_g,2,0,4,0.00,40425.00
6,products,product_length_cm,2,0,0,7.00,105.00
7,products,product_height_cm,2,0,0,2.00,105.00
8,products,product_width_cm,2,0,0,6.00,118.00


In [32]:
business_rule_checks = {
    "negative_item_price": (
        clean_tables["order_items"]["price"] < 0
    ),
    "negative_freight_value": (
        clean_tables["order_items"]["freight_value"] < 0
    ),
    "negative_payment_value": (
        clean_tables["payments"]["payment_value"] < 0
    ),
    "payment_installments_less_than_one": (
        clean_tables["payments"][
            "payment_installments"
        ] < 1
    ),
    "review_score_outside_1_to_5": (
        ~clean_tables["reviews"][
            "review_score"
        ].between(1, 5)
    ),
    "nonpositive_product_weight": (
        clean_tables["products"][
            "product_weight_g"
        ].notna()
        & (
            clean_tables["products"][
                "product_weight_g"
            ] <= 0
        )
    ),
    "nonpositive_product_length": (
        clean_tables["products"][
            "product_length_cm"
        ].notna()
        & (
            clean_tables["products"][
                "product_length_cm"
            ] <= 0
        )
    ),
    "nonpositive_product_height": (
        clean_tables["products"][
            "product_height_cm"
        ].notna()
        & (
            clean_tables["products"][
                "product_height_cm"
            ] <= 0
        )
    ),
    "nonpositive_product_width": (
        clean_tables["products"][
            "product_width_cm"
        ].notna()
        & (
            clean_tables["products"][
                "product_width_cm"
            ] <= 0
        )
    ),
}

business_rule_checks_df = pd.DataFrame({
    "check_name": list(business_rule_checks.keys()),
    "affected_rows": [
        int(mask.fillna(False).sum())
        for mask in business_rule_checks.values()
    ],
})

display(business_rule_checks_df)

,check_name,affected_rows
0,negative_item_price,0
1,negative_freight_value,0
2,negative_payment_value,0
3,payment_installments_less_than_one,2
4,review_score_outside_1_to_5,0
5,nonpositive_product_weight,4
6,nonpositive_product_length,0
7,nonpositive_product_height,0
8,nonpositive_product_width,0


In [33]:
invalid_installments_df = (
    clean_tables["payments"]
    .loc[
        clean_tables["payments"][
            "payment_installments"
        ] < 1
    ]
    .copy()
)

display(invalid_installments_df)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [34]:
invalid_product_weight_df = (
    clean_tables["products"]
    .loc[
        clean_tables["products"][
            "product_weight_g"
        ].notna()
        & (
            clean_tables["products"][
                "product_weight_g"
            ] <= 0
        ),
        [
            "product_id",
            "product_category_name",
            "product_category_name_english",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ],
    ]
    .copy()
)

display(invalid_product_weight_df)

,product_id,product_category_name,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0


In [35]:
payments_clean = clean_tables["payments"].copy()
products_clean = clean_tables["products"].copy()

invalid_installment_mask = (
    payments_clean["payment_installments"] < 1
)

payments_clean.loc[
    invalid_installment_mask,
    "payment_installments",
] = 1

invalid_weight_mask = (
    products_clean["product_weight_g"].notna()
    & (
        products_clean["product_weight_g"] <= 0
    )
)

products_clean.loc[
    invalid_weight_mask,
    "product_weight_g",
] = np.nan

clean_tables["payments"] = payments_clean
clean_tables["products"] = products_clean

print(
    "Corrected installment rows:",
    int(invalid_installment_mask.sum()),
)

print(
    "Weights changed to missing:",
    int(invalid_weight_mask.sum()),
)

Corrected installment rows: 2
Weights changed to missing: 4


In [36]:
remaining_invalid_installments = int(
    (
        clean_tables["payments"][
            "payment_installments"
        ] < 1
    ).sum()
)

remaining_nonpositive_weights = int(
    (
        clean_tables["products"][
            "product_weight_g"
        ].notna()
        & (
            clean_tables["products"][
                "product_weight_g"
            ] <= 0
        )
    ).sum()
)

assert remaining_invalid_installments == 0
assert remaining_nonpositive_weights == 0

print("Numeric anomaly treatment passed.")

Numeric anomaly treatment passed.


In [37]:
orders_clean = clean_tables["orders"].copy()

timestamp_rule_names = [
    "approval_before_purchase",
    "carrier_before_purchase",
    "carrier_before_approval",
    "delivery_before_carrier",
    "delivery_before_purchase",
    "estimated_before_purchase",
]

status_rule_names = [
    "delivered_status_missing_delivery_date",
    "non_delivered_status_with_delivery_date",
]

timestamp_anomaly_mask = pd.concat(
    [
        order_timestamp_checks[rule_name]
        for rule_name in timestamp_rule_names
    ],
    axis=1,
).fillna(False).any(axis=1)

status_date_mismatch_mask = pd.concat(
    [
        order_timestamp_checks[rule_name]
        for rule_name in status_rule_names
    ],
    axis=1,
).fillna(False).any(axis=1)

orders_clean["has_timestamp_anomaly"] = (
    timestamp_anomaly_mask
)

orders_clean["has_status_date_mismatch"] = (
    status_date_mismatch_mask
)

orders_clean["is_valid_delivery_record"] = (
    orders_clean["order_status"].eq("delivered")
    & orders_clean[
        "order_delivered_customer_date"
    ].notna()
    & ~orders_clean["has_timestamp_anomaly"]
)

clean_tables["orders"] = orders_clean

print(
    "Orders with timestamp anomalies:",
    orders_clean["has_timestamp_anomaly"].sum(),
)

print(
    "Orders with status/date mismatch:",
    orders_clean["has_status_date_mismatch"].sum(),
)

print(
    "Valid delivery records:",
    orders_clean["is_valid_delivery_record"].sum(),
)

Orders with timestamp anomalies: 1382
Orders with status/date mismatch: 14
Valid delivery records: 95097


In [38]:
treatment_summary_df = pd.DataFrame({
    "table_name": [
        "payments",
        "products",
        "orders",
    ],
    "issue": [
        "payment_installments_less_than_one",
        "nonpositive_product_weight",
        "timestamp_or_status_anomalies",
    ],
    "affected_rows": [
        int(invalid_installment_mask.sum()),
        int(invalid_weight_mask.sum()),
        int(
            (
                timestamp_anomaly_mask
                | status_date_mismatch_mask
            ).sum()
        ),
    ],
    "treatment": [
        "Changed installment count from 0 to 1",
        "Changed zero weight to missing",
        "Preserved records and added quality flags",
    ],
})

order_status_df.to_csv(
    reports_dir / "order_status_counts.csv",
    index=False,
    encoding="utf-8",
)

order_timestamp_checks_df.to_csv(
    reports_dir / "order_timestamp_checks.csv",
    index=False,
    encoding="utf-8",
)

numeric_checks_df.to_csv(
    reports_dir / "numeric_validity_checks.csv",
    index=False,
    encoding="utf-8",
)

business_rule_checks_df.to_csv(
    reports_dir / "business_rule_checks.csv",
    index=False,
    encoding="utf-8",
)

treatment_summary_df.to_csv(
    reports_dir / "anomaly_treatments.csv",
    index=False,
    encoding="utf-8",
)

print("Anomaly reports saved.")

Anomaly reports saved.


In [40]:
clean_tables["geolocation"] = (
    geolocation_clean.copy()
)

print(
    clean_tables["geolocation"].shape
)

(19015, 6)


In [41]:
cleaning_summary_records = []

for table_name in raw_tables:
    raw_df = raw_tables[table_name]
    clean_df = clean_tables[table_name]

    cleaning_summary_records.append({
        "table_name": table_name,
        "raw_rows": len(raw_df),
        "clean_rows": len(clean_df),
        "row_change": len(clean_df) - len(raw_df),
        "raw_columns": len(raw_df.columns),
        "clean_columns": len(clean_df.columns),
    })

cleaning_summary_df = pd.DataFrame(
    cleaning_summary_records
)

display(cleaning_summary_df)

,table_name,raw_rows,clean_rows,row_change,raw_columns,clean_columns
0,customers,99441,99441,0,5,5
1,geolocation,1000163,19015,-981148,5,6
2,order_items,112650,112650,0,7,7
3,payments,103886,103886,0,5,5
4,reviews,99224,99224,0,7,7
5,orders,99441,99441,0,8,11
6,products,32951,32951,0,9,10
7,sellers,3095,3095,0,4,4
8,category_translation,71,74,3,2,2


In [42]:
final_key_map = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": [
        "product_category_name"
    ],
    "order_items": [
        "order_id",
        "order_item_id",
    ],
    "payments": [
        "order_id",
        "payment_sequential",
    ],
    "reviews": [
        "review_id",
        "order_id",
    ],
    "geolocation": [
        "geolocation_zip_code_prefix"
    ],
}

In [43]:
final_key_records = []

for table_name, key_columns in final_key_map.items():
    df = clean_tables[table_name]

    null_key_rows = int(
        df[key_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    duplicate_key_rows = int(
        df.duplicated(
            subset=key_columns,
            keep=False,
        ).sum()
    )

    final_key_records.append({
        "table_name": table_name,
        "key_columns": " + ".join(key_columns),
        "null_key_rows": null_key_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "key_valid": (
            null_key_rows == 0
            and duplicate_key_rows == 0
        ),
    })

final_key_checks_df = pd.DataFrame(
    final_key_records
)

display(final_key_checks_df)

,table_name,key_columns,null_key_rows,duplicate_key_rows,key_valid
0,customers,customer_id,0,0,True
1,orders,order_id,0,0,True
2,products,product_id,0,0,True
3,sellers,seller_id,0,0,True
4,category_translation,product_category_name,0,0,True
5,order_items,order_id + order_item_id,0,0,True
6,payments,order_id + payment_sequential,0,0,True
7,reviews,review_id + order_id,0,0,True
8,geolocation,geolocation_zip_code_prefix,0,0,True


In [44]:
# 1. Date conversion must not introduce parsing failures
assert date_conversion_df[
    "parse_failures"
].eq(0).all()

# 2. Every product must have an English category
assert clean_tables["products"][
    "product_category_name_english"
].isna().sum() == 0

# 3. The geolocation table must contain one row per ZIP-code prefix
assert clean_tables["geolocation"][
    "geolocation_zip_code_prefix"
].duplicated().sum() == 0

assert clean_tables[
    "geolocation"
].shape == (19015, 6)

# 4. Payment installment counts must be at least one
assert (
    clean_tables["payments"][
        "payment_installments"
    ] < 1
).sum() == 0

# 5. Non-missing product weights must be greater than zero
assert (
    clean_tables["products"][
        "product_weight_g"
    ].dropna() <= 0
).sum() == 0

# 6. Order-quality flags must not contain missing values
order_quality_columns = [
    "has_timestamp_anomaly",
    "has_status_date_mismatch",
    "is_valid_delivery_record",
]

assert clean_tables["orders"][
    order_quality_columns
].isna().sum().sum() == 0

# 7. Row counts must remain unchanged for the main business tables
unchanged_row_tables = [
    "customers",
    "order_items",
    "payments",
    "reviews",
    "orders",
    "products",
    "sellers",
]

for table_name in unchanged_row_tables:
    assert (
        len(clean_tables[table_name])
        == len(raw_tables[table_name])
    )

print("All final cleaning checks passed.")

All final cleaning checks passed.


In [45]:
processed_file_map = {
    "customers": "customers_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "order_items": "order_items_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": (
        "category_translation_clean.csv"
    ),
}

processed_inventory_records = []

for table_name, file_name in processed_file_map.items():
    output_path = processed_dir / file_name
    df = clean_tables[table_name]

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d %H:%M:%S",
    )

    processed_inventory_records.append({
        "table_name": table_name,
        "file_name": file_name,
        "rows": len(df),
        "columns": len(df.columns),
        "file_size_mb": round(
            output_path.stat().st_size
            / (1024 ** 2),
            2,
        ),
    })

processed_inventory_df = pd.DataFrame(
    processed_inventory_records
)

display(processed_inventory_df)

print("All processed datasets were saved.")

,table_name,file_name,rows,columns,file_size_mb
0,customers,customers_clean.csv,99441,5,8.19
1,geolocation,geolocation_clean.csv,19015,6,0.81
2,order_items,order_items_clean.csv,112650,7,14.22
3,payments,payments_clean.csv,103886,5,5.37
4,reviews,reviews_clean.csv,99224,7,13.29
5,orders,orders_clean.csv,99441,11,18.22
6,products,products_clean.csv,32951,10,3.11
7,sellers,sellers_clean.csv,3095,4,0.16
8,category_translation,category_translation_clean.csv,74,2,0.00


All processed datasets were saved.


In [47]:
saved_processed_files = list(
    processed_dir.glob("*_clean.csv")
)

assert len(saved_processed_files) == 9

print(
    "Number of processed files:",
    len(saved_processed_files),
)

for file_path in sorted(saved_processed_files):
    print(file_path.name)

Number of processed files: 9
category_translation_clean.csv
customers_clean.csv
geolocation_clean.csv
order_items_clean.csv
orders_clean.csv
payments_clean.csv
products_clean.csv
reviews_clean.csv
sellers_clean.csv


In [46]:
cleaning_summary_df.to_csv(
    reports_dir / "cleaning_summary.csv",
    index=False,
    encoding="utf-8",
)

final_key_checks_df.to_csv(
    reports_dir / "final_clean_key_checks.csv",
    index=False,
    encoding="utf-8",
)

processed_inventory_df.to_csv(
    reports_dir / "processed_data_inventory.csv",
    index=False,
    encoding="utf-8",
)

print("Final cleaning reports saved.")

Final cleaning reports saved.


In [48]:
geo_raw = raw_tables["geolocation"]
geo_clean = clean_tables["geolocation"]

valid_raw_coordinates = (
    geo_raw["geolocation_lat"].between(-90, 90)
    & geo_raw["geolocation_lng"].between(-180, 180)
)

geolocation_summary_df = pd.DataFrame({
    "metric": [
        "raw_rows",
        "exact_duplicate_rows",
        "invalid_coordinate_rows",
        "unique_zip_codes_before",
        "clean_rows",
        "duplicate_zip_codes_after",
    ],
    "value": [
        len(geo_raw),
        int(geo_raw.duplicated().sum()),
        int((~valid_raw_coordinates).sum()),
        int(
            geo_raw[
                "geolocation_zip_code_prefix"
            ].nunique()
        ),
        len(geo_clean),
        int(
            geo_clean[
                "geolocation_zip_code_prefix"
            ].duplicated().sum()
        ),
    ],
})

geo_zip_values = set(
    geo_clean["geolocation_zip_code_prefix"]
)

zip_coverage_records = []

for table_name, column_name in {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
}.items():
    zip_series = clean_tables[
        table_name
    ][column_name].dropna()

    unmatched_mask = ~zip_series.isin(
        geo_zip_values
    )

    zip_coverage_records.append({
        "table_name": table_name,
        "zip_column": column_name,
        "total_rows": len(
            clean_tables[table_name]
        ),
        "unmatched_rows": int(
            unmatched_mask.sum()
        ),
        "unmatched_unique_zip_codes": int(
            zip_series[
                unmatched_mask
            ].nunique()
        ),
        "coverage_pct": round(
            (
                1
                - unmatched_mask.sum()
                / len(zip_series)
            )
            * 100,
            2,
        ),
    })

zip_coverage_df = pd.DataFrame(
    zip_coverage_records
)

geolocation_summary_df.to_csv(
    reports_dir
    / "geolocation_cleaning_checks.csv",
    index=False,
    encoding="utf-8",
)

zip_coverage_df.to_csv(
    reports_dir
    / "geolocation_zip_coverage.csv",
    index=False,
    encoding="utf-8",
)

print("Geolocation reports saved.")

Geolocation reports saved.
